In [ ]:

import cv2
import os

# Path to the folder containing MP4 videos
input_folder = 'C:\\Users\\bsef0\\Downloads\\Vedios'
output_folder = 'C:\\Users\\bsef0\\Downloads\\Vedios\\Frames'

# Make sure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Get all MP4 files from the input folder
video_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.mp4')]
total_count=0

for video_file in video_files:
    video_path = os.path.join(input_folder, video_file)
    video_name = os.path.splitext(video_file)[0]

    # Create subfolder for each video's frames
    video_output_folder = os.path.join(output_folder, video_name)
    os.makedirs(video_output_folder, exist_ok=True)

    # Open video file
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        # Save frame as JPG image
        frame_filename = os.path.join(video_output_folder, f'frame_{total_count:05d}.jpg')
        cv2.imwrite(frame_filename, frame)
        frame_count += 1
        total_count += 1

    cap.release()
    print(f"Saved {frame_count} frames from '{video_file}' to '{video_output_folder}'")


In [11]:
import cv2
import torch
import numpy as np
from torchvision.transforms import Compose, Normalize

midas_transforms = Compose([
    lambda x: cv2.resize(x, (384, 384)),  # Adjust size depending on model
    lambda x: x / 255.0,
    lambda x: np.transpose(x, (2, 0, 1)),
    lambda x: torch.from_numpy(x).float(),
    Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


In [ ]:
def save_depth_prediction(img_path, model, device, save_path):
    img = cv2.imread(img_path)[:, :, ::-1]  # BGR to RGB
    img = cv2.resize(img, (320, 224))       # Resize to match input size

    img_input = midas_transforms(img).unsqueeze(0).to(device)




    with torch.no_grad():
        prediction = model(img_input)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(224, 320),
            mode="bicubic",
            align_corners=False,
        ).squeeze()

    depth_map = prediction.cpu().numpy()
    depth_min = depth_map.min()
    depth_max = depth_map.max()
    normalized = (255 * (depth_map - depth_min) / (depth_max - depth_min)).astype(np.uint8)

    cv2.imwrite(save_path, normalized)


In [21]:
import os

In [ ]:
#temp code

folder_path ='C:\\Users\\bsef0\\Downloads\\Vedios\\Frames\\Potholes'
# Loop through and remove .png files
for filename in os.listdir(folder_path):
    if filename.endswith(".png"):
        file_path = os.path.join(folder_path, filename)
        os.remove(file_path)
        print(f"Deleted: {file_path}")

In [2]:
import os
import torch
import cv2
import argparse
from midas.dpt_depth import DPTDepthModel
from midas.transforms import Resize, NormalizeImage, PrepareForNet
import torchvision.transforms as T
from PIL import Image


c:\Users\bsef0\Downloads\MonoDepth\MiDaS-master\MiDaS-master\venvMidas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\bsef0\Downloads\MonoDepth\MiDaS-master\MiDaS-master\venvMidas\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [8]:


import torch
import os
import cv2
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

input_folder = 'C:\\Users\\bsef0\\Downloads\\MonoDepth\\MiDaS-master\\MiDaS-master\\dataset\\Potholes_Raw_New'
output_folder = 'C:\\Users\\bsef0\\Downloads\\MonoDepth\\MiDaS-master\\MiDaS-master\\dataset\\depth_maps'
os.makedirs(output_folder, exist_ok=True)



In [9]:
import torch

# Valid model type
model_type = "DPT_Large"

# Load model and transforms
midas = torch.hub.load("intel-isl/MiDaS", model_type)
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
midas.eval()


Using cache found in C:\Users\bsef0/.cache\torch\hub\intel-isl_MiDaS_master
Using cache found in C:\Users\bsef0/.cache\torch\hub\intel-isl_MiDaS_master


DPTDepthModel(
  (pretrained): Module(
    (model): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_featur

In [10]:
import os

for img_file in os.listdir("dataset/Potholes_Raw_New"):
    img_path = os.path.join("dataset/Potholes_Raw_New", img_file)
    
    # Get the file name without extension
    filename_wo_ext, ext = os.path.splitext(img_file)

    # Create new file name with _depth postfix
    depth_filename = f"{filename_wo_ext}_depth.png"
    save_path = os.path.join("dataset/depth_maps/", depth_filename)
    #print(save_path)

    save_depth_prediction(img_path, midas, device, save_path)



TypeError: 'module' object is not callable